# Notebook 03 — Hyperparameter Tuning

Systematic hyperparameter optimization using `RandomizedSearchCV` on the 80% development set with 5-fold stratified CV. Primary metric for optimization is accuracy.

## 2. Environment & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import os
import itertools

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings('ignore')

## 3. Load & Prepare Data

We load the `german.data` dataset, define column names, map the target variable (1->1 Good, 2->0 Bad), and perform an 80/20 stratified split. We also define the preprocessors for tree-based and neural network models.

In [ ]:
RANDOM_SEED = 42

# Column names and feature types
column_names = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 
    'savings_status', 'employment', 'installment_commitment', 'personal_status', 
    'other_parties', 'residence_since', 'property_magnitude', 'age', 
    'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 
    'own_telephone', 'foreign_worker', 'target'
]

numerical_cols = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
categorical_cols = ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']

# Load data
data_path = '../data/german.data'
df = pd.read_csv(data_path, sep=' ', names=column_names, header=None)

# Convert target: 1->1 (Good), 2->0 (Bad)
df['target'] = df['target'].map({1: 1, 2: 0})

# Split into features and target
X = df.drop('target', axis=1)
y = df['target']

# 80/20 stratified split for development and holdout testing
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

print(f"X_dev shape: {X_dev.shape}")
print(f"y_dev shape: {y_dev.shape}")

In [ ]:
# Preprocessor for tree-based models (Ordinal Encoding for categorical)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ])

# Preprocessor for linear/NN models (Standardization + One-Hot Encoding)
mlp_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

## 4. CV Strategy

Using a 5-fold stratified cross-validation strategy.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

## 5. XGBoost Tuning

Defining a randomized search space for XGBoost over trees, depth, learning rate, subsampling, and regularization.

In [ ]:
xgb_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('classifier', XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss', use_label_encoder=False))
])

xgb_param_dist = {
    'classifier__n_estimators': randint(50, 500),
    'classifier__max_depth': randint(3, 10),
    'classifier__learning_rate': uniform(0.01, 0.29),  # 0.01 to 0.30
    'classifier__subsample': uniform(0.6, 0.4),  # 0.6 to 1.0
    'classifier__colsample_bytree': uniform(0.6, 0.4),
    'classifier__min_child_weight': randint(1, 10),
    'classifier__gamma': uniform(0, 0.5),
    'classifier__reg_alpha': uniform(0, 1.0),
    'classifier__reg_lambda': uniform(0.5, 1.5),
}

print("Starting XGBoost tuning...")
start_time = time.time()
xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_param_dist, n_iter=50, cv=cv, 
    scoring='accuracy', random_state=RANDOM_SEED, n_jobs=-1, verbose=1
)
xgb_search.fit(X_dev, y_dev)
xgb_time = time.time() - start_time

print(f"XGBoost tuning time: {xgb_time:.2f} seconds")
print(f"Best XGBoost CV Accuracy: {xgb_search.best_score_:.4f}")
print("Best XGBoost Params:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")

## 6. CatBoost Tuning

CatBoost handles categorical variables natively, but because we specify `cat_features` at fit time (or directly in the dataset), we'll write a manual random search loop to evaluate configurations via cross-validation.

In [ ]:
import random

catboost_param_dist = {
    'iterations': [100, 200, 300, 500],
    'depth': [4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'random_strength': [0.5, 1.0, 2.0, 4.0],
    'bagging_temperature': [0.0, 0.5, 1.0, 2.0],
}

def sample_params(param_dist):
    return {k: random.choice(v) for k, v in param_dist.items()}

n_iter_cb = 50
random.seed(RANDOM_SEED)
sampled_params_list = [sample_params(catboost_param_dist) for _ in range(n_iter_cb)]

best_cb_score = 0
best_catboost_params = None

print("Starting CatBoost manual tuning...")
start_time = time.time()

for i, params in enumerate(sampled_params_list):
    fold_scores = []
    for train_idx, val_idx in cv.split(X_dev, y_dev):
        X_train_fold, X_val_fold = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
        y_train_fold, y_val_fold = y_dev.iloc[train_idx], y_dev.iloc[val_idx]
        
        cb = CatBoostClassifier(**params, random_seed=RANDOM_SEED, cat_features=categorical_cols, verbose=0)
        cb.fit(X_train_fold, y_train_fold)
        preds = cb.predict(X_val_fold)
        fold_scores.append(accuracy_score(y_val_fold, preds))
    
    mean_score = np.mean(fold_scores)
    if mean_score > best_cb_score:
        best_cb_score = mean_score
        best_catboost_params = params

cb_time = time.time() - start_time

print(f"CatBoost tuning time: {cb_time:.2f} seconds")
print(f"Best CatBoost CV Accuracy: {best_cb_score:.4f}")
print("Best CatBoost Params:")
for k, v in best_catboost_params.items():
    print(f"  {k}: {v}")

## 7. Random Forest Tuning

Tuning Random Forest hyperparameters like number of estimators, depth, split criteria, and feature subset sizes.

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_SEED))
])

rf_param_dist = {
    'classifier__n_estimators': randint(100, 500),
    'classifier__max_depth': [None, 5, 10, 15, 20, 25],
    'classifier__min_samples_split': randint(2, 20),
    'classifier__min_samples_leaf': randint(1, 10),
    'classifier__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'classifier__class_weight': [None, 'balanced'],
}

print("Starting Random Forest tuning...")
start_time = time.time()
rf_search = RandomizedSearchCV(
    rf_pipeline, rf_param_dist, n_iter=50, cv=cv,
    scoring='accuracy', random_state=RANDOM_SEED, n_jobs=-1, verbose=1
)
rf_search.fit(X_dev, y_dev)
rf_time = time.time() - start_time

print(f"Random Forest tuning time: {rf_time:.2f} seconds")
print(f"Best RF CV Accuracy: {rf_search.best_score_:.4f}")
print("Best RF Params:")
for k, v in rf_search.best_params_.items():
    print(f"  {k}: {v}")

## 8. MLP Tuning

Tuning Neural Network (MLP) architecture, learning rate, regularization, and optimization parameters.

In [ ]:
mlp_pipeline = Pipeline([
    ('preprocessor', mlp_preprocessor),
    ('classifier', MLPClassifier(random_state=RANDOM_SEED, early_stopping=True))
])

mlp_param_dist = {
    'classifier__hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 50, 25), (200, 100), (64, 32)],
    'classifier__learning_rate_init': uniform(0.0005, 0.0095),
    'classifier__alpha': uniform(0.0001, 0.01),
    'classifier__batch_size': [16, 32, 64, 128],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam', 'sgd'],
    'classifier__max_iter': [500, 1000],
}

print("Starting MLP tuning...")
start_time = time.time()
mlp_search = RandomizedSearchCV(
    mlp_pipeline, mlp_param_dist, n_iter=50, cv=cv,
    scoring='accuracy', random_state=RANDOM_SEED, n_jobs=-1, verbose=1
)
mlp_search.fit(X_dev, y_dev)
mlp_time = time.time() - start_time

print(f"MLP tuning time: {mlp_time:.2f} seconds")
print(f"Best MLP CV Accuracy: {mlp_search.best_score_:.4f}")
print("Best MLP Params:")
for k, v in mlp_search.best_params_.items():
    print(f"  {k}: {v}")

## 9. Tuning Results Comparison

Evaluate all optimized models using multiple metrics on CV.

In [ ]:
scoring = ['accuracy', 'roc_auc', 'precision', 'recall', 'f1']
results = []

def evaluate_model(model, name, is_catboost=False):
    if is_catboost:
        # Manual CV evaluation for CatBoost to get multiple metrics
        scores = {s: [] for s in scoring}
        from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
        for train_idx, val_idx in cv.split(X_dev, y_dev):
            X_train_fold, X_val_fold = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
            y_train_fold, y_val_fold = y_dev.iloc[train_idx], y_dev.iloc[val_idx]
            
            model.fit(X_train_fold, y_train_fold)
            preds = model.predict(X_val_fold)
            probas = model.predict_proba(X_val_fold)[:, 1]
            
            scores['accuracy'].append(accuracy_score(y_val_fold, preds))
            scores['roc_auc'].append(roc_auc_score(y_val_fold, probas))
            scores['precision'].append(precision_score(y_val_fold, preds))
            scores['recall'].append(recall_score(y_val_fold, preds))
            scores['f1'].append(f1_score(y_val_fold, preds))
        
        res = {'Model': name}
        for s in scoring:
            col_name = 'Best CV Accuracy' if s == 'accuracy' else s.replace('_', '-').upper() if s == 'roc_auc' else s.capitalize()
            res[col_name] = np.mean(scores[s])
        return res
    else:
        cv_results = cross_validate(model, X_dev, y_dev, cv=cv, scoring=scoring)
        return {
            'Model': name,
            'Best CV Accuracy': np.mean(cv_results['test_accuracy']),
            'ROC-AUC': np.mean(cv_results['test_roc_auc']),
            'Precision': np.mean(cv_results['test_precision']),
            'Recall': np.mean(cv_results['test_recall']),
            'F1': np.mean(cv_results['test_f1'])
        }

# Evaluate tuned models
results.append(evaluate_model(xgb_search.best_estimator_, 'XGBoost (Tuned)'))

cb_best = CatBoostClassifier(**best_catboost_params, random_seed=RANDOM_SEED, cat_features=categorical_cols, verbose=0)
results.append(evaluate_model(cb_best, 'CatBoost (Tuned)', is_catboost=True))

results.append(evaluate_model(rf_search.best_estimator_, 'Random Forest (Tuned)'))
results.append(evaluate_model(mlp_search.best_estimator_, 'MLP (Tuned)'))

tuned_results_df = pd.DataFrame(results)
print(tuned_results_df)

os.makedirs('../results/tables', exist_ok=True)
tuned_results_df.to_csv('../results/tables/tuned_cv_results.csv', index=False)

## 10. Save Best Hyperparameters

Saving the best hyperparameters as a JSON file for use in final evaluation.

In [ ]:
def convert_to_python_types(d):
    res = {}
    for k, v in d.items():
        new_key = k.replace('classifier__', '')
        if hasattr(v, 'item'): # numpy types
            res[new_key] = v.item()
        else:
            res[new_key] = v
    return res

best_params = {
    'XGBoost': convert_to_python_types(xgb_search.best_params_),
    'CatBoost': convert_to_python_types(best_catboost_params),
    'Random Forest': convert_to_python_types(rf_search.best_params_),
    'MLP': convert_to_python_types(mlp_search.best_params_)
}

with open('../results/tables/best_hyperparameters.json', 'w') as f:
    json.dump(best_params, f, indent=2, default=str)

print("Best parameters saved.")

## 11. Visualization

Compare baseline vs tuned accuracy for each model.

In [ ]:
os.makedirs('../results/plots', exist_ok=True)

baseline_path = '../results/tables/baseline_cv_results.csv'
if os.path.exists(baseline_path):
    baseline_df = pd.read_csv(baseline_path)
    # Match model names if necessary, assuming format "XGBoost" vs "XGBoost (Tuned)"
    if 'Accuracy' in baseline_df.columns:
        baseline_df['Model_Base'] = baseline_df['Model'].apply(lambda x: x.replace(' (Baseline)', ''))
        tuned_results_df['Base_Model'] = tuned_results_df['Model'].apply(lambda x: x.replace(' (Tuned)', ''))
        
        merged_df = pd.merge(baseline_df[['Model_Base', 'Accuracy']], tuned_results_df[['Base_Model', 'Best CV Accuracy']], 
                             left_on='Model_Base', right_on='Base_Model', how='inner')
        merged_df = merged_df.rename(columns={'Accuracy': 'Baseline Accuracy', 'Best CV Accuracy': 'Tuned Accuracy'})
        
        # Plotting
        fig, ax = plt.subplots(figsize=(10, 6))
        x = np.arange(len(merged_df['Model_Base']))
        width = 0.35
        
        ax.bar(x - width/2, merged_df['Baseline Accuracy'], width, label='Baseline')
        ax.bar(x + width/2, merged_df['Tuned Accuracy'], width, label='Tuned')
        
        ax.set_ylabel('CV Accuracy')
        ax.set_title('Baseline vs Tuned CV Accuracy')
        ax.set_xticks(x)
        ax.set_xticklabels(merged_df['Model_Base'])
        ax.legend()
        
        plt.ylim(0.5, 1.0)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig('../results/plots/tuning_comparison.png')
        plt.show()
else:
    print("Baseline results not found or formatted differently. Generating standalone tuning plot.")
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(x='Best CV Accuracy', y='Model', data=tuned_results_df, ax=ax)
    ax.set_title('Tuned Model CV Accuracies')
    plt.tight_layout()
    plt.savefig('../results/plots/tuning_comparison.png')
    plt.show()

## 12. Summary

**Observations:**
- Hyperparameter tuning generally yields incremental improvements over baselines.
- Random Forest and CatBoost often display strong performance even before extensive tuning, but fine-tuning parameters like max_depth, learning rate, and feature sampling usually helps models like XGBoost to avoid overfitting.
- Depending on the exact splits and randomness, accuracy mostly remains in the 75%-80% range, indicating that the signal in the German Credit dataset is quite noisy.
- We will proceed with the best configurations identified here to evaluate on the final test set in Notebook 04.